# 08 - Generate All Tables and Figures
Generates publication-ready LaTeX tables and PDF figures.

Includes:
- Tables 2–5 (ours) + Table 6 (SOTA) + Table 7 (distance)
- Fig 2 (pipeline), Fig 3 (qualitative), Fig 4 (quantitative bar),
  Fig 5 (failure cases), Fig SOTA comparison

**Prerequisite**: Run notebook 05 or 09 first.

In [1]:
import sys, os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import load_config, ensure_dirs
from src.experiments.summarize import summarize_results
from src.visualization.plot_utils import (
    set_paper_style, plot_quantitative_summary, plot_sota_comparison,
    plot_failure_cases, plot_ablation_boxplots,
)
from src.visualization.mesh_renderer import (
    render_mesh_comparison, render_pipeline_figure,
)

set_paper_style()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'chair_leg.yaml'))
ensure_dirs(cfg)

metrics_dir = os.path.join(cfg['paths']['output_dir'], 'metrics')
figures_dir = cfg['paths']['figures_dir']
tables_dir = os.path.join(cfg['paths']['output_dir'], 'tables')
os.makedirs(figures_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)

df = pd.read_csv(os.path.join(metrics_dir, 'all_results.csv'))
print(f'Loaded {len(df)} samples')

Loaded 84 samples


In [3]:
# ============================================================
# Generate ALL tables
# ============================================================
tables = summarize_results(df, cfg['paths']['output_dir'])

for name, tbl in tables.items():
    print(f'\n{"=" * 70}')
    print(f'{name}')
    print(f'{"=" * 70}')
    print(tbl.to_string(index=False, float_format='%.4f'))
    
    # Save LaTeX
    latex_path = os.path.join(tables_dir, f'{name}.tex')
    tbl.to_latex(latex_path, index=False, float_format='%.4f',
                 caption=name.replace('_', ' ').title(),
                 label=f'tab:{name}')
    print(f'  -> {latex_path}')


table2_quantitative
          Method  Residual ↓  Improvement ↑  New Vtx ↓  New Faces ↓  Avg Quality ↑  Min Quality ↑
      Center-fan      0.0000        39.9517    43.7500    1761.3690         0.2671         0.0725
    Planar + RPA      1.9384        38.0133     0.0000    1656.9405         0.4341         0.1333
Planar + LH-only     38.2569         1.6948     0.0000      96.1905         0.4122         0.2016
  -> D:\MyJupyter\Works\3DPART\outputs\tables\table2_quantitative.tex

table3_locality
          Method  Inside ↑  Outside ↓  Locality ↑
      Center-fan 1065.2976   696.0714      0.6656
    Planar + RPA 1013.6786   643.2619      0.6710
Planar + LH-only   55.2857    40.9048      0.5717
Trimesh fill-all    8.4524    58.2500      0.0657
 Adv-front + RPA  920.1250   505.4792      0.7174
   Poisson recon    0.0000     0.0000      0.0000
  -> D:\MyJupyter\Works\3DPART\outputs\tables\table3_locality.tex

table4_failures
        ID  RP Residual  LH Residual  LH Outside  RP Improv  LH Imp

In [4]:
# ============================================================
# Figure 4: Quantitative summary (3-panel, ours)
# ============================================================
plot_quantitative_summary(
    df, os.path.join(figures_dir, 'fig4_quantitative_summary.pdf'),
    methods=None
)
print('Saved fig4_quantitative_summary.pdf')

Saved fig4_quantitative_summary.pdf


In [5]:
# ============================================================
# Figure: SOTA comparison (if SOTA data available)
# ============================================================
all_methods = ['center_fan', 'planar_removed_part_aware', 'planar_largest_hole_only',
               'trimesh_fill_all', 'advancing_front_rpa', 'open3d_poisson']
available = [m for m in all_methods
             if f'{m}/target_loop_length_after' in df.columns
             and df[f'{m}/target_loop_length_after'].notna().sum() > 0]

if len(available) > 3:
    plot_sota_comparison(df, available,
                        os.path.join(figures_dir, 'fig_sota_comparison.pdf'))
    print(f'Saved fig_sota_comparison.pdf ({len(available)} methods)')
else:
    print('SOTA methods not available. Run notebook 09 first for SOTA figures.')

Saved fig_sota_comparison.pdf (6 methods)


In [6]:
# ============================================================
# Figure 5: Failure case bar chart
# ============================================================
plot_failure_cases(df, os.path.join(figures_dir, 'fig5_failure_cases.pdf'))
print('Saved fig5_failure_cases.pdf')

Saved fig5_failure_cases.pdf


In [7]:
# ============================================================
# Figure 2: Pipeline figure + Figure 3: Qualitative comparison
# ============================================================
from src.data.sample_loader import SampleLoader
from src.geometry.boundary import extract_boundary_loops
from src.target_selection.selectors import select_target_loops_by_bbox, select_largest_loop
from src.repair.center_fan import center_fan_repair
from src.repair.planar_patch import planar_triangulation_repair

loader = SampleLoader(cfg['paths']['raw_data_dir'])
margin = cfg['repair']['margin']
prox_thresh = cfg['repair']['proximity_threshold']

# Pick representative samples
from src.data.dataset_index import DatasetIndex
index = DatasetIndex(cfg['paths']['raw_data_dir'])
sample_ids = index.sample_ids[:3]  # First 3 for qualitative

for sid in sample_ids:
    try:
        sample = loader.load(str(sid))
        damaged = sample['damaged_mesh']
        removed = sample['removed_part_mesh']
        loops = extract_boundary_loops(damaged)
        targets_rpa = select_target_loops_by_bbox(damaged, loops, removed, margin, prox_thresh)
        targets_lh = select_largest_loop(damaged, loops)

        # Repair with each method
        res_cf = center_fan_repair(damaged, targets_rpa)
        res_planar = planar_triangulation_repair(damaged, targets_rpa)
        res_lh = planar_triangulation_repair(damaged, targets_lh)

        # Fig 2: Pipeline
        render_pipeline_figure(
            damaged, removed, res_planar['repaired_mesh'],
            targets_rpa, loops, res_planar['new_faces'],
            os.path.join(figures_dir, f'fig2_pipeline_{sid}.pdf'),
        )

        # Fig 3: Side-by-side qualitative comparison
        render_mesh_comparison(
            meshes=[damaged, res_cf['repaired_mesh'],
                    res_planar['repaired_mesh'], res_lh['repaired_mesh']],
            titles=['$M_d$', 'Center-fan', 'Planar+RPA', 'Planar+LH-only'],
            save_path=os.path.join(figures_dir, f'fig3_qualitative_{sid}.pdf'),
            patch_faces_list=[
                None,
                res_cf['new_faces'],
                res_planar['new_faces'],
                res_lh['new_faces'],
            ],
            boundary_loops_list=[loops, None, None, None],
        )
        print(f'  Sample {sid}: pipeline + qualitative figures saved')

        del res_cf, res_planar, res_lh
    except Exception as e:
        print(f'  Error on {sid}: {e}')

  Sample 2882: pipeline + qualitative figures saved
  Sample 40584: pipeline + qualitative figures saved
  Sample 42201: pipeline + qualitative figures saved


In [8]:
# ============================================================
# Summary of all generated files
# ============================================================
print('\n' + '=' * 60)
print('Generated Files')
print('=' * 60)
for d in [figures_dir, tables_dir]:
    print(f'\n{d}:')
    if os.path.exists(d):
        for f in sorted(os.listdir(d)):
            size = os.path.getsize(os.path.join(d, f))
            print(f'  {f} ({size:,} bytes)')


Generated Files

D:\MyJupyter\Works\3DPART\outputs\figures:
  .ipynb_checkpoints (0 bytes)
  ablation_scatter.pdf (14,187 bytes)
  ablation_target_selection.pdf (19,924 bytes)
  boundary_loop_stats.pdf (55,179 bytes)
  fig2_pipeline_2882.pdf (456,409 bytes)
  fig2_pipeline_40584.pdf (446,249 bytes)
  fig2_pipeline_42201.pdf (445,487 bytes)
  fig3_qualitative_2882.pdf (591,184 bytes)
  fig3_qualitative_40584.pdf (578,380 bytes)
  fig3_qualitative_42201.pdf (583,457 bytes)
  fig4_quantitative_summary.pdf (29,113 bytes)
  fig5_failure_cases.pdf (19,271 bytes)
  fig_learning_comparison.pdf (26,322 bytes)
  fig_sota_comparison.pdf (22,089 bytes)
  multi_category_comparison.pdf (19,446 bytes)
  renders (16,384 bytes)
  repair_center_fan_2882.pdf (33,939 bytes)
  repair_largest_hole_2882.pdf (28,824 bytes)
  repair_planar_2882.pdf (34,081 bytes)
  target_selection_agreement.pdf (18,216 bytes)

D:\MyJupyter\Works\3DPART\outputs\tables:
  .ipynb_checkpoints (0 bytes)
  table2_quantitative.csv 